# Customer Segmentation Using Clustering

Discover actionable customer groups from recency, frequency, monetary and engagement behaviour.

**Portfolio category:** Clustering

**Data mode:** Verified demo mode

This notebook keeps labels out of fitting wherever labels exist, uses deterministic seeds,
reports unsupervised-specific diagnostics, and avoids hard-coded results.

## 1. Project setup

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

## 2. RFM-style customer data

In [ ]:
segment_sizes = [310, 250, 240, 200]
prototypes = [
    [15, 18, 1800, 24, 0.02],
    [65, 9, 850, 14, 0.05],
    [150, 3, 260, 5, 0.12],
    [35, 26, 3200, 32, 0.03],
]
rows = []
for segment, (size, proto) in enumerate(zip(segment_sizes, prototypes)):
    recency, frequency, monetary, engagement, return_rate = proto
    rows.append(pd.DataFrame({
        "recency_days": np.clip(rng.normal(recency, recency * 0.25 + 3, size), 1, None),
        "purchase_frequency": np.clip(rng.normal(frequency, 3, size), 1, None),
        "monetary_value": np.clip(rng.lognormal(np.log(monetary), 0.28, size), 20, None),
        "digital_engagement": np.clip(rng.normal(engagement, 4, size), 0, 40),
        "return_rate": np.clip(rng.normal(return_rate, 0.02, size), 0, 0.4),
        "hidden_segment": segment,
    }))
customers = pd.concat(rows, ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
feature_names = ["recency_days", "purchase_frequency", "monetary_value", "digital_engagement", "return_rate"]
display(customers.head())

## 3. Data quality and distributions

In [ ]:
display(customers[feature_names].describe().T)
print("Missing cells:", int(customers[feature_names].isna().sum().sum()))
customers[feature_names].hist(figsize=(12, 7), bins=25)
plt.tight_layout()

## 4. Robust feature preparation

In [ ]:
scaler = RobustScaler()
X = scaler.fit_transform(customers[feature_names])

## 5. Choose the number of clusters with multiple metrics

In [ ]:
rows = []
for k in range(2, 9):
    candidate = KMeans(n_clusters=k, n_init=25, random_state=RANDOM_STATE)
    candidate_labels = candidate.fit_predict(X)
    rows.append({
        "k": k,
        "silhouette": silhouette_score(X, candidate_labels),
        "davies_bouldin": davies_bouldin_score(X, candidate_labels),
        "calinski_harabasz": calinski_harabasz_score(X, candidate_labels),
    })
scores = pd.DataFrame(rows)
display(scores.round(3))
best_k = int(scores.sort_values(["silhouette", "davies_bouldin"], ascending=[False, True]).iloc[0]["k"])
model = KMeans(n_clusters=best_k, n_init=40, random_state=RANDOM_STATE)
customers["cluster"] = model.fit_predict(X)
print("Selected clusters:", best_k)

## 6. Stability check

In [ ]:
stability = []
for seed in [7, 19, 31, 43, 71]:
    alternative = KMeans(n_clusters=best_k, n_init=20, random_state=seed).fit_predict(X)
    stability.append(adjusted_rand_score(customers["cluster"], alternative))
display(pd.Series(stability, name="adjusted_rand_vs_reference").describe().to_frame())

## 7. Segment profiles

In [ ]:
profile = customers.groupby("cluster")[feature_names].mean()
profile["customers"] = customers.groupby("cluster").size()
display(profile.round(2))
sns.heatmap(profile[feature_names].apply(lambda col: (col - col.mean()) / col.std()), cmap="vlag", center=0)
plt.title("Standardised segment profiles")
plt.tight_layout()

## 8. Two-dimensional projection

In [ ]:
projection = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)
sns.scatterplot(x=projection[:, 0], y=projection[:, 1], hue=customers["cluster"], palette="tab10", s=35)
plt.title("Customer segments in PCA space")
plt.tight_layout()

## 9. Key findings

Name segments only after reading their profiles. Cluster IDs are arbitrary and should never be treated as ordered customer value.

## 10. Interpretation and responsible use

Treat the output as exploratory evidence, not ground truth. For customer segmentation using clustering,
validate stability on newer data, inspect edge cases, and review domain risks before
turning clusters, rankings or anomaly scores into decisions.

## 11. Next steps

- Replace demonstration data with a versioned, licensed dataset.
- Track data quality, drift and stability across repeated runs.
- Add domain-specific review before deployment.
- Package inference only after reproducibility and privacy checks pass.

All numeric results are generated at execution time; none are hard-coded.